In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("OlistETL").master("local[*]").getOrCreate()
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/01 23:13:20 WARN Utils: Your hostname, mrnobody, resolves to a loopback address: 127.0.1.1; using 192.168.0.47 instead (on interface wlp3s0)
26/09/01 23:13:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 23:13:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.1.1'

In [2]:
from pyspark.sql.types import StructType,StructField,StringType,TimestampType
order_schema = StructType([
StructField("order_id",StringType()),
StructField("customer_id",StringType()),
StructField("order_status",StringType()),
StructField("order_purchase_timestamp",TimestampType()),
StructField("order_approved_at",TimestampType()),
StructField("order_delivered_carrier_date",TimestampType()),
StructField("order_delivered_customer_date",TimestampType()),
StructField("order_estimated_delivery_date",TimestampType())

])

In [3]:
orders_df = spark.read.format("csv").option("header",True).load("../data/raw/olist_orders_dataset.csv",schema=order_schema)

In [4]:
orders_df.printSchema()
orders_df.show(5,truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [5]:
#1. Total row count
orders_df.count()

99441

In [6]:
#2. number of null id values
from pyspark.sql.functions import col
orders_df.filter(col('order_id').isNull()).count()


0

In [7]:
orders_df.select('order_id').distinct().count()

99441

In [8]:
orders_df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    invoiced|
|     created|
|   delivered|
| unavailable|
|  processing|
|    approved|
+------------+



In [9]:
##checking null value for one column

from pyspark.sql.functions import when,sum
orders_df.agg(sum(when(col("order_approved_at").isNull(),1).otherwise(0)).alias("null value in order approved at")).show()


+-------------------------------+
|null value in order approved at|
+-------------------------------+
|                            160|
+-------------------------------+



In [10]:
##checking null value for all column
null_checks = [sum(when(col(column_name).isNull(),1).otherwise(0)).alias(column_name)
                for column_name in orders_df.columns ]

orders_df.agg(*null_checks).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [11]:
orders_df.filter(col('order_delivered_customer_date').isNull()).groupBy(col('order_status')).count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  619|
|    invoiced|  314|
|     created|    5|
|   delivered|    8|
| unavailable|  609|
|  processing|  301|
|    approved|    2|
+------------+-----+



In [12]:
orders_df.filter(col("order_delivered_customer_date").isNull() & (col("order_status")=="delivered")).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b...|ec05a6d8558c6455f...|   delivered|     2017-11-28 17:44:07|2017-11-28 17:56:40|         2017-11-30 18:12:23|                         NULL|          2017-12-18 00:00:00|
|f5dd62b788049ad9f...|5e89028e024b381dc...|   delivered|     2018-06-20 06:58:43|2018-06-20 07:19:05|         2018-06-25 08:05:00|                         NULL|          2018-07-16 00:00:00|
|2ebdfc4f15f23b914...|29f0540231702fda0...|  

In [13]:
orders_df.filter(col("order_purchase_timestamp")>col("order_approved_at")).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|a68a639ac918e408e...|e0a720da8f7baff1f...|   delivered|     2018-03-25 02:59:41|2018-03-25 02:15:23|         2018-03-27 20:19:02|          2018-04-11 19:56:28|          2018-05-03 00:00:00|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+



In [14]:
orders_df.filter(col("order_delivered_customer_date")<col("order_delivered_carrier_date")).show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|a1abeb653a4d4cd1e142ccb8c82cd069|5f50465da00b7fed5dd1239f4ecf6e2c|delivered   |2017-07-20 11:20:52     |2017-07-21 06:43:14|2017-07-28 16:57:58         |2017-07-25 19:32:56          |2017-08-14 00:00:00          |
|383aa8b2724fe452d9ccd9934a8c628b|b1cb2f9d7a19480f3749e248db14d58f|delivered   |2017-07-02 20:58:43     |2017-07-02 21:10:20|2017-07-07 17:2

In [15]:
from pyspark.sql.types import IntegerType,DoubleType 
order_item_schema = StructType([
StructField("order_id",StringType()),
StructField("order_item_id",IntegerType()),     
StructField("product_id",StringType()),      
StructField("seller_id",StringType()),      
StructField("shipping_limit_date",TimestampType()),  
StructField("price",DoubleType()),      
StructField("freight_value",DoubleType())

])


In [16]:
order_items_df = spark.read.format("csv").option("header",True).load('../data/raw/olist_order_items_dataset.csv',schema=order_item_schema)

In [17]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [18]:
order_items_df.show(5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9|19.93        |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.0|17.87        |
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa

In [19]:
order_items_df.count()

112650

In [20]:
order_items_df.select("order_id").distinct().count()

98666

In [21]:
order_items_df.groupBy("order_id").count().filter(col('count')>1).orderBy(col("count").desc()).show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|1b15974a0141d54e3...|   20|
|ab14fdcfbe524636d...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|73c8ab38f07dc9438...|   14|
|9bdc4d4c71aa1de46...|   14|
|37ee401157a3a0b28...|   13|
|2c2a19b5703863c90...|   12|
|af822dacd6f5cff73...|   12|
|3a213fcdfe7d98be7...|   12|
|637617b3ffe9e2f7a...|   12|
|c05d6a79e55da72ca...|   12|
|71dab1155600756af...|   11|
|5a3b1c29a49756e75...|   11|
|7f2c22c54cbae5509...|   11|
|6c355e2913545fa6f...|   11|
|9aec4e1ae90b23c7b...|   10|
|30bdf3d824d824610...|   10|
|ca3625898fbd48669...|   10|
+--------------------+-----+
only showing top 20 rows


In [22]:
order_items_df.select('order_id','order_item_id').distinct().count()

112650

In [23]:
order_items_df.select("order_id","order_item_id").count()

112650

In [24]:
orders_without_item_count = orders_df.join(order_items_df,on="order_id",how="anti").count()

In [25]:
item_without_order = order_items_df.join(orders_df,on="order_id",how="anti").count()

In [26]:
orders_without_item = orders_df.join(order_items_df,on="order_id",how="anti")

In [27]:
orders_without_item.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped|    1|
|    canceled|  164|
|     created|    5|
| unavailable|  603|
|    invoiced|    2|
+------------+-----+



In [28]:
customer_schema = StructType([
StructField("customer_id",StringType()),
StructField("customer_unique_id",StringType()),
StructField("customer_zip_code_prefix",StringType()),
StructField("customer_city",StringType()),
StructField("customer_state",StringType())])


product_schema = StructType([
StructField( "product_id",StringType()),
StructField("product_category_name",StringType()),
StructField("product_name_lenght",IntegerType()),
StructField("product_description_lenght",IntegerType()),
StructField("product_photos_qty",IntegerType()),
StructField("product_weight_g",IntegerType()),
StructField("product_length_cm",IntegerType()),
StructField("product_height_cm",IntegerType()),
StructField("product_width_cm",IntegerType())
])

customer_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .schema(schema=customer_schema) \
    .load("../data/raw/olist_customers_dataset.csv")
product_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .schema(schema=product_schema) \
    .load("../data/raw/olist_products_dataset.csv")

In [29]:
customer_df.count()
product_df.count()

32951

In [30]:
customer_df.printSchema()
product_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [31]:
customer_df.show(5)
product_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                   08775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows
+--------------------+---------

In [32]:
customer_null_check = customer_df.withColumn("null count",when(col("customer_id").isNull(),1).otherwise(0)) 

In [33]:
customer_null_check.show(truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+----------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|null count|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+----------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |0         |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |sao bernardo do campo|SP            |0         |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151                   |sao paulo            |SP            |0         |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|08775                   |mogi das cruzes      |SP            |0         |
|4f2d8ab171c8

In [34]:
customer_df.filter(col("customer_id").isNull()).count()

0

In [35]:
product_df.filter(col("product_id").isNull()).count()

0

In [36]:
customer_df.select(col("customer_id")).distinct().count()

99441

In [37]:
customer_df.select(col("customer_unique_id")).distinct().count()

96096

In [38]:
product_df.select(col("product_id")).distinct().count()

32951

In [39]:
product_df.select("product_id").count()

32951

In [40]:
from pyspark.sql.types import DecimalType
seller_schema = StructType([
StructField("seller_id",StringType()),
StructField("seller_zip_code_prefix", StringType()),
StructField("seller_city",StringType()),
StructField("seller_state",StringType())
])

payment_schema=StructType([
StructField("order_id",StringType()),
StructField("payment_sequential",IntegerType()),
StructField("payment_type",StringType()),
StructField("payment_installments", IntegerType()),
StructField("payment_value",DecimalType(18,2))
])

review_schema = StructType([
StructField("review_id",StringType()),
StructField("order_id",StringType()),
StructField("review_score",IntegerType()),
StructField("review_comment_title",StringType()),
StructField("review_comment_message",StringType()),
StructField("review_creation_date",TimestampType()),
StructField("review_answer_timestamp", TimestampType())
])

geolocation_schema = StructType([
StructField("geolocation_zip_code_prefix",StringType()),
StructField("geolocation_lat",DoubleType()),
StructField("geolocation_lng",DoubleType()),
StructField("geolocation_city",StringType()),
StructField("geolocation_state",StringType())

])

category_translation_schema = StructType([
StructField("product_category_name",StringType()),
StructField("product_category_name_english",StringType())
])

In [41]:
seller_df = spark.read.format("csv").option("header",True).load("../data/raw/olist_sellers_dataset.csv",schema=seller_schema)
payment_df = spark.read.format("csv").option("header",True).load("../data/raw/olist_order_payments_dataset.csv",schema=payment_schema)
review_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .schema(review_schema) \
    .load("../data/raw/olist_order_reviews_dataset.csv")
geolocation_df = spark.read.format("csv").option("header",True).load("../data/raw/olist_geolocation_dataset.csv",schema=geolocation_schema)
category_translation_df = spark.read.format("csv").option("header",True).load("../data/raw/product_category_name_translation.csv",schema=category_translation_schema)

In [42]:
print(seller_df.count())
print(payment_df.count())
print(review_df.count())
print(geolocation_df.count())
print(category_translation_df.count())

3095
103886
99224
1000163
71


In [43]:
def profile_table(df,key_columns):
    total_row = df.count()
    distinct_key = df.select(*key_columns).distinct().count()
    duplicate_key = total_row-distinct_key
    null_checks = [
    sum(
        when(col(column_name).isNull(), 1)
        .otherwise(0)
    ).alias(column_name)

    for column_name in df.columns
]
    print(total_row,distinct_key,duplicate_key)

    df.agg(*null_checks).show()    

profile_table(orders_df,["order_id"])
profile_table(order_items_df,["order_id","order_item_id"])
profile_table(customer_df, ["customer_id"])

profile_table(product_df, ["product_id"])

profile_table(seller_df, ["seller_id"])


99441 99441 0
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

112650 112650 0
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_i

In [44]:
profile_table(
    payment_df,
    ["order_id", "payment_sequential"]
)

103886 103886 0
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



In [45]:
profile_table(
    review_df,
    ["review_id"]
)

99224 98410 814
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|               87656|                 58247|                   0|                      0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [46]:
review_df.filter(
    col("review_id").isNull()
).show(truncate=False)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [47]:
review_df.filter(
    col("review_score").isNull()
).show(10, truncate=False)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [48]:
duplicate_reviews = (
    review_df
    .groupBy("review_id")
    .count()
    .filter(col("count") > 1)
)

In [49]:
duplicate_review_rows = review_df.join(
    duplicate_reviews.select("review_id"),
    on="review_id",
    how="inner"
)

In [50]:
duplicate_review_rows.show(10, truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+---------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                             |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+---------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|28642ce6250b94cc72bc85960aec6c62|e239d280236cdd3c40cb2c033f681d1c|5           |NULL                |NULL                                                                                               |2018-03-25 00:00:00 |2018-03-25 21:03:02    |
|a0a641414ff

In [51]:
duplicate_review_rows \
    .orderBy("review_id") \
    .show(20, truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                                    |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------

In [52]:
duplicate_review_rows \
    .filter(col("review_id") == "28642ce6250b94cc72bc85960aec6c62") \
    .show(truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|28642ce6250b94cc72bc85960aec6c62|e239d280236cdd3c40cb2c033f681d1c|5           |NULL                |NULL                  |2018-03-25 00:00:00 |2018-03-25 21:03:02    |
|28642ce6250b94cc72bc85960aec6c62|bc42a955f289870d5789e6e437206300|5           |NULL                |NULL                  |2018-03-25 00:00:00 |2018-03-25 21:03:02    |
+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----

In [53]:
profile_table(
    review_df,
    ["review_id", "order_id"]
)

99224 99224 0
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|               87656|                 58247|                   0|                      0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [54]:
geolocation_df.select(
    "geolocation_zip_code_prefix"
).distinct().count()

19015

In [55]:
geolocation_df.select(
    "geolocation_lat",
    "geolocation_lng"
).distinct().count()

718463

In [56]:
geolocation_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng"
).distinct().count()

720154

In [57]:
geolocation_df.distinct().count()

738332

In [58]:
profile_table(
    category_translation_df,
    ["product_category_name"]
)

71 71 0
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



In [59]:
order_silver_df= orders_df.withColumn("Invalid_approval_sequence",when(col("order_purchase_timestamp")>col("order_approved_at"),1).otherwise(0))

In [60]:
order_silver_df.groupBy("invalid_approval_sequence").count().show()

+-------------------------+-----+
|invalid_approval_sequence|count|
+-------------------------+-----+
|                        0|99440|
|                        1|    1|
+-------------------------+-----+



In [61]:
order_silver_df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|Invalid_approval_sequence|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|                        0|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|  

In [62]:
order_silver_df = order_silver_df.withColumn("Invalid_delivery_sequence",when(col("order_delivered_customer_date").isNull() | col("order_delivered_carrier_date").isNull(),None).when(col("order_delivered_customer_date")<col("order_delivered_carrier_date"),1).otherwise(0))

In [63]:
order_silver_df.groupBy("Invalid_delivery_sequence").count().show()

+-------------------------+-----+
|Invalid_delivery_sequence|count|
+-------------------------+-----+
|                     NULL| 2966|
|                        1|   23|
|                        0|96452|
+-------------------------+-----+



In [64]:
import sys
sys.path.append("../src")

In [65]:
from ingestion import ingest_olist_data

In [66]:
df = ingest_olist_data(spark=spark,raw_path_to_file="olist_orders_dataset.csv",file_schema=order_schema)
df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [67]:
df.count()

99441

In [68]:
olist_config = {
    "orders": {
        "filename": "olist_orders_dataset.csv",
        "schema": order_schema
    },
    "customers":{
        "filename":"olist_customers_dataset.csv",
        "schema":customer_schema
    },
    "geolocation":{
            "filename":"olist_geolocation_dataset.csv",
            "schema":geolocation_schema
        },
    "order_items":{
            "filename":"olist_order_items_dataset.csv",
            "schema":order_item_schema
        },
    "order_payment":{
            "filename":"olist_order_payments_dataset.csv",
            "schema":payment_schema
        },
    "orders_reviews":{
                    "filename":"olist_order_reviews_dataset.csv",
                    "schema":review_schema
                },
    "products":{
            "filename":"olist_products_dataset.csv",
            "schema":product_schema
        },
    "sellers":{
            "filename":"olist_sellers_dataset.csv",
            "schema":seller_schema
        },
    "category_name_translation":{
            "filename":"product_category_name_translation.csv",
            "schema":category_translation_schema
        },

    }
# df = ingest_olist_data(spark=spark,raw_path_to_file=olist_config["orders"]["filename"],file_schema=olist_config["orders"]["schema"])
# df.count()

In [69]:
for name, config in olist_config.items():
    print(name)
    print(config["filename"])

orders
olist_orders_dataset.csv
customers
olist_customers_dataset.csv
geolocation
olist_geolocation_dataset.csv
order_items
olist_order_items_dataset.csv
order_payment
olist_order_payments_dataset.csv
orders_reviews
olist_order_reviews_dataset.csv
products
olist_products_dataset.csv
sellers
olist_sellers_dataset.csv
category_name_translation
product_category_name_translation.csv


In [70]:
import logging
dataframe ={}
logging.basicConfig(level= logging.INFO)
for name,config in olist_config.items():
    try:
        df = ingest_olist_data(spark=spark,raw_path_to_file=config["filename"],file_schema=config["schema"])
        dataframe[name]=df 
    except Exception as e:
        print(f"failed to ingest {config['filename']}:{e}")

INFO:ingestion:Reading file :olist_orders_dataset.csv
INFO:ingestion:Successfully read file: olist_orders_dataset.csv
INFO:ingestion:Reading file :olist_customers_dataset.csv
INFO:ingestion:Successfully read file: olist_customers_dataset.csv
INFO:ingestion:Reading file :olist_geolocation_dataset.csv
INFO:ingestion:Successfully read file: olist_geolocation_dataset.csv
INFO:ingestion:Reading file :olist_order_items_dataset.csv
INFO:ingestion:Successfully read file: olist_order_items_dataset.csv
INFO:ingestion:Reading file :olist_order_payments_dataset.csv
INFO:ingestion:Successfully read file: olist_order_payments_dataset.csv
INFO:ingestion:Reading file :olist_order_reviews_dataset.csv
INFO:ingestion:Successfully read file: olist_order_reviews_dataset.csv
INFO:ingestion:Reading file :olist_products_dataset.csv
INFO:ingestion:Successfully read file: olist_products_dataset.csv
INFO:ingestion:Reading file :olist_sellers_dataset.csv
INFO:ingestion:Successfully read file: olist_sellers_datase

In [71]:
dataframe.keys()

dict_keys(['orders', 'customers', 'geolocation', 'order_items', 'order_payment', 'orders_reviews', 'products', 'sellers', 'category_name_translation'])

In [72]:
dataframe["order_items"].count()

112650

In [73]:
for name in dataframe:
    print(f"{name}:{ dataframe[name].count()}")

orders:99441
customers:99441
geolocation:1000163
order_items:112650
order_payment:103886
orders_reviews:104162
products:32951
sellers:3095
category_name_translation:71


In [74]:
dataframe['orders'].groupBy("order_id").count().filter(col('count')>1).show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [75]:
order_null_check = dataframe["orders"].withColumn("null_count",when(col("order_id").isNull(),1).otherwise(0))
order_null_check.agg(sum("null_count").alias("order_id_null_checks")).show()
dataframe['orders'].columns

+--------------------+
|order_id_null_checks|
+--------------------+
|                   0|
+--------------------+



['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [76]:
dataframe["orders"].agg(
    sum(
        when(col("order_id").isNull(), 1).otherwise(0)
    ).alias("order_id_null_count"),

    sum(
        when(col("customer_id").isNull(), 1).otherwise(0)
    ).alias("customer_id_null_count"),
    sum(
        when(col("order_status").isNull(), 1).otherwise(0)
        ).alias("order_status_null_count"),
    sum(
        when(col("order_purchase_timestamp").isNull(), 1).otherwise(0)
        ).alias("order_purchase_timestamp_null_count"),

).show()

+-------------------+----------------------+-----------------------+-----------------------------------+
|order_id_null_count|customer_id_null_count|order_status_null_count|order_purchase_timestamp_null_count|
+-------------------+----------------------+-----------------------+-----------------------------------+
|                  0|                     0|                      0|                                  0|
+-------------------+----------------------+-----------------------+-----------------------------------+



In [77]:
null_expressions=[]
for column_name in dataframe["orders"].columns:
    null_check = (sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(f"{column_name}_null_count"))
    null_expressions.append(null_check)


In [78]:
print(*null_expressions)

Column<'sum(CASE WHEN isNull(order_id) THEN 1 ELSE 0 END) AS order_id_null_count'> Column<'sum(CASE WHEN isNull(customer_id) THEN 1 ELSE 0 END) AS customer_id_null_count'> Column<'sum(CASE WHEN isNull(order_status) THEN 1 ELSE 0 END) AS order_status_null_count'> Column<'sum(CASE WHEN isNull(order_purchase_timestamp) THEN 1 ELSE 0 END) AS order_purchase_timestamp_null_count'> Column<'sum(CASE WHEN isNull(order_approved_at) THEN 1 ELSE 0 END) AS order_approved_at_null_count'> Column<'sum(CASE WHEN isNull(order_delivered_carrier_date) THEN 1 ELSE 0 END) AS order_delivered_carrier_date_null_count'> Column<'sum(CASE WHEN isNull(order_delivered_customer_date) THEN 1 ELSE 0 END) AS order_delivered_customer_date_null_count'> Column<'sum(CASE WHEN isNull(order_estimated_delivery_date) THEN 1 ELSE 0 END) AS order_estimated_delivery_date_null_count'>


In [79]:
dataframe['orders'].agg(*null_expressions).show()

+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|order_id_null_count|customer_id_null_count|order_status_null_count|order_purchase_timestamp_null_count|order_approved_at_null_count|order_delivered_carrier_date_null_count|order_delivered_customer_date_null_count|order_estimated_delivery_date_null_count|
+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|                  0|                     0|                      0|                                  0|                         160|                                   1783|                                    2965|                  

In [80]:
from validation import null_checks
null_result = null_checks(dataframe['orders'])
null_result.show()

+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|order_id_null_count|customer_id_null_count|order_status_null_count|order_purchase_timestamp_null_count|order_approved_at_null_count|order_delivered_carrier_date_null_count|order_delivered_customer_date_null_count|order_estimated_delivery_date_null_count|
+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|                  0|                     0|                      0|                                  0|                         160|                                   1783|                                    2965|                  

In [81]:
from validation import duplicate_checks
df_duplicate_check = duplicate_checks(dataframe["orders"],"order_id")
df_duplicate_check.show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [82]:
dataframe['orders'].filter(col("order_purchase_timestamp") > col("order_approved_at")).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|a68a639ac918e408e...|e0a720da8f7baff1f...|   delivered|     2018-03-25 02:59:41|2018-03-25 02:15:23|         2018-03-27 20:19:02|          2018-04-11 19:56:28|          2018-05-03 00:00:00|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+



In [83]:
invalid_delivery_dates = dataframe["orders"].filter(col("order_delivered_customer_date") < col("order_delivered_carrier_date"))
invalid_delivery_dates.count()

23

In [84]:
order_delivered_before_purchase = dataframe["orders"].filter(col("order_delivered_customer_date") < col("order_purchase_timestamp"))
order_delivered_before_purchase.count()

0

In [85]:
from validation import date_validation
invalid_dates = date_validation(dataframe["orders"],"order_delivered_carrier_date","order_delivered_customer_date")
invalid_dates.count()

23

In [86]:
dataframe["orders"].select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    invoiced|
|     created|
|   delivered|
| unavailable|
|  processing|
|    approved|
+------------+



In [87]:
valid_status = [
"shipped",
"canceled",
"invoiced",
"created",
"delivered",
"unavailable",
"processing",
"approved"
]

from validation import valid_values_check

invalid_status =valid_values_check(dataframe['orders'],"order_status",valid_status)
invalid_status.count()


0

In [88]:
from validation import referential_integrity_check

invalid_customer = referential_integrity_check(dataframe['orders'],dataframe["customers"],"customer_id")
invalid_customer.count()

0

In [89]:
from transformation import add_delivery_days
order_transformed = add_delivery_days(dataframe['orders'])
null_dates = order_transformed.filter(col("order_delivered_customer_date").isNull())
null_dates.select("order_status",
"order_delivered_customer_date",
"delivery_days").show()

+------------+-----------------------------+-------------+
|order_status|order_delivered_customer_date|delivery_days|
+------------+-----------------------------+-------------+
|    invoiced|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|    invoiced|                         NULL|         NULL|
|  processing|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
| unavailable|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|  processing|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|    canceled|                         NULL|         NUL

In [90]:
null_dates = order_transformed.filter(col("order_delivered_customer_date").isNull())
null_dates.select("order_status",
"order_delivered_customer_date",
"delivery_days").show()

+------------+-----------------------------+-------------+
|order_status|order_delivered_customer_date|delivery_days|
+------------+-----------------------------+-------------+
|    invoiced|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|    invoiced|                         NULL|         NULL|
|  processing|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
| unavailable|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|  processing|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|     shipped|                         NULL|         NULL|
|    canceled|                         NULL|         NUL

In [91]:
from pyspark.sql.functions import year, month
order_transformed = order_transformed.withColumns({"purchase_year": year(col("order_purchase_timestamp")),"purchase_month":month(col("order_purchase_timestamp"))})
order_transformed.select("order_purchase_timestamp",
"purchase_year",
"purchase_month").show()

+------------------------+-------------+--------------+
|order_purchase_timestamp|purchase_year|purchase_month|
+------------------------+-------------+--------------+
|     2017-10-02 10:56:33|         2017|            10|
|     2018-07-24 20:41:37|         2018|             7|
|     2018-08-08 08:38:49|         2018|             8|
|     2017-11-18 19:28:06|         2017|            11|
|     2018-02-13 21:18:39|         2018|             2|
|     2017-07-09 21:57:05|         2017|             7|
|     2017-04-11 12:22:08|         2017|             4|
|     2017-05-16 13:10:30|         2017|             5|
|     2017-01-23 18:29:09|         2017|             1|
|     2017-07-29 11:55:02|         2017|             7|
|     2017-05-16 19:41:10|         2017|             5|
|     2017-07-13 19:58:11|         2017|             7|
|     2018-06-07 10:06:19|         2018|             6|
|     2018-07-25 17:44:10|         2018|             7|
|     2018-03-01 14:14:28|         2018|        

In [96]:
from transformation import add_date_parts
orders_transformed = add_delivery_days(dataframe["orders"])

orders_transformed = add_date_parts(
    orders_transformed,
    "order_purchase_timestamp",
    "purchase"
)
order_transformed.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------+--------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|purchase_year|purchase_month|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------+--------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|            8|         2017|            10|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     201

In [111]:
from pyspark.sql.functions import lit
order_transformed = order_transformed.withColumn("is_late",when(col("order_delivered_customer_date").isNull(),lit(None)).when(col("order_estimated_delivery_date")<col("order_delivered_customer_date"),1).otherwise(0))
order_transformed.select('order_delivered_customer_date',"order_estimated_delivery_date","is_late").show()


+-----------------------------+-----------------------------+-------+
|order_delivered_customer_date|order_estimated_delivery_date|is_late|
+-----------------------------+-----------------------------+-------+
|          2017-10-10 21:25:13|          2017-10-18 00:00:00|      0|
|          2018-08-07 15:27:45|          2018-08-13 00:00:00|      0|
|          2018-08-17 18:06:29|          2018-09-04 00:00:00|      0|
|          2017-12-02 00:28:42|          2017-12-15 00:00:00|      0|
|          2018-02-16 18:17:02|          2018-02-26 00:00:00|      0|
|          2017-07-26 10:57:55|          2017-08-01 00:00:00|      0|
|                         NULL|          2017-05-09 00:00:00|   NULL|
|          2017-05-26 12:55:51|          2017-06-07 00:00:00|      0|
|          2017-02-02 14:08:10|          2017-03-06 00:00:00|      0|
|          2017-08-16 17:14:30|          2017-08-23 00:00:00|      0|
|          2017-05-29 11:18:31|          2017-06-07 00:00:00|      0|
|          2017-07-1

In [112]:
order_transformed.filter(
    col("order_delivered_customer_date").isNull()
).select(
    "order_status",
    "order_delivered_customer_date",
    "is_late"
).show()

+------------+-----------------------------+-------+
|order_status|order_delivered_customer_date|is_late|
+------------+-----------------------------+-------+
|    invoiced|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|    invoiced|                         NULL|   NULL|
|  processing|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
| unavailable|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|  processing|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|    canceled|                         NULL|   NULL|
|     shipped|                         NULL|   NULL|
|     shipped|                         NULL|  

In [113]:
dataframe["order_items"].printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [115]:
dataframe["order_items"].show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [121]:
from pyspark.sql.functions import count
order_items_agg = dataframe["order_items"].groupBy("order_id").agg(count("order_item_id").alias("order_item_count"),sum("price").alias("total_item_value"),sum("freight_value").alias("total_freight"))
order_items_agg.show(5)

+--------------------+----------------+----------------+-------------+
|            order_id|order_item_count|total_item_value|total_freight|
+--------------------+----------------+----------------+-------------+
|014405982914c2cde...|               2|           49.23|         29.2|
|019886de8f385a39b...|               1|           159.9|         28.5|
|01a6ad782455876aa...|               1|           34.99|         15.1|
|01d907b3e209269e1...|               1|          151.99|        17.77|
|028dc52e12ddda803...|               1|           49.99|        11.73|
+--------------------+----------------+----------------+-------------+
only showing top 5 rows


In [124]:
order_items_duplicate_check = duplicate_checks(order_items_agg,'order_id')
order_items_duplicate_check.count()

0

In [125]:
order_items_agg.count()

98666

In [129]:
orders_without_items = dataframe["orders"].join(order_items_agg,on="order_id",how="left_anti")
orders_without_items.count()
orders_without_items.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|8e24261a7e58791d1...|64a254d30eed42cd0...| unavailable|     2017-11-16 15:09:28|2017-11-16 15:26:57|                        NULL|                         NULL|          2017-12-05 00:00:00|
|c272bcd21c287498b...|9582c5bbecc65eb56...| unavailable|     2018-01-31 11:31:37|2018-01-31 14:23:50|                        NULL|                         NULL|          2018-02-16 00:00:00|
|37553832a3a89c9b2...|7607cd563696c27ed...| u

In [131]:
orders_without_items.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped|    1|
|    canceled|  164|
|     created|    5|
| unavailable|  603|
|    invoiced|    2|
+------------+-----+



In [133]:
order_curated = order_transformed.join(order_items_agg,on="order_id",how="left")
order_curated.count()

99441

In [134]:
dataframe["order_payment"].printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(18,2) (nullable = true)



In [139]:
dataframe["order_payment"].show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [143]:
payment_duplicate = duplicate_checks(dataframe['order_payment'],"order_id")
payment_duplicate.show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8ca5bdac5ebe8f2d6...|    9|
|54066aeaaf3ac32e7...|    2|
|f44cb69655f8e4d13...|    2|
|a9b929204be626d61...|    2|
|0c077fbe69b84abe1...|    2|
|cd05a1fc2781f43f5...|    2|
|b60cab8c479e1e804...|    2|
|7a5472f7c8cecc2e1...|    2|
|3ba1f632cf89a08ca...|    2|
|35ab20ce8b706d545...|    2|
|3f4f6a378519479cb...|    2|
|5ded9a59e8920225f...|    2|
|a1e28dc56f8cf4e56...|    2|
|251f0a3981c4a8cb8...|    5|
|1826d2a2eb6ba6e3e...|    2|
|30e934394c047a409...|    2|
|85ff97380814f8f14...|    2|
|f63a31c3349b87273...|    2|
|8dd9758206f8d9c23...|    2|
|41d85a7a138b7205e...|    2|
+--------------------+-----+
only showing top 20 rows


In [147]:
payment_agg = dataframe['order_payment'].groupBy("order_id").agg(sum("payment_value").alias("total_payment_value"),count("payment_value").alias("payment_count"))

In [149]:
payment_agg.show()

+--------------------+-------------------+-------------+
|            order_id|total_payment_value|payment_count|
+--------------------+-------------------+-------------+
|bb2d7e3141540afc2...|              37.15|            1|
|85be7c94bcd3f908f...|              72.75|            1|
|8ca5bdac5ebe8f2d6...|             189.08|            9|
|54066aeaaf3ac32e7...|             148.06|            2|
|5db54d41d5ebd6d76...|             136.26|            1|
|41537821ce113ccef...|              89.27|            1|
|beca5b5e9460824d8...|             239.50|            1|
|33f1e992ba3e439bf...|             248.51|            1|
|3fa59277573f0fe06...|              91.05|            1|
|e239d280236cdd3c4...|             102.03|            1|
|81d715926d69a15e7...|             315.89|            1|
|3e654c7f3c4a852f6...|             136.93|            1|
|66b9c991ee308f934...|            3358.24|            1|
|c1101a911387fc42d...|              99.88|            1|
|281f470497aa1a0af...|         

In [151]:
order_without_payment = dataframe["orders"].join(payment_agg,on="order_id",how="left_anti")
order_without_payment.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|bfbd0f9bdef843021...|86dc2ffce2dfff336...|   delivered|     2016-09-15 12:16:38|2016-09-15 12:16:38|         2016-11-07 17:11:53|          2016-11-09 07:47:38|          2016-10-04 00:00:00|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+



In [152]:
order_curated = order_curated.join(payment_agg,on="order_id",how="left")
order_curated.count()

99441

In [153]:
order_curated.select(
    "order_id",
    "order_item_count",
    "total_item_value",
    "total_freight",
    "total_payment_value",
    "payment_count"
).show(5)

+--------------------+----------------+----------------+-------------+-------------------+-------------+
|            order_id|order_item_count|total_item_value|total_freight|total_payment_value|payment_count|
+--------------------+----------------+----------------+-------------+-------------------+-------------+
|e481f51cbdc54678b...|               1|           29.99|         8.72|              38.71|            3|
|53cdb2fc8bc7dce0b...|               1|           118.7|        22.76|             141.46|            1|
|47770eb9100c2d0c4...|               1|           159.9|        19.22|             179.12|            1|
|949d5b44dbf5de918...|               1|            45.0|         27.2|              72.20|            1|
|ad21c59c0840e6cb8...|               1|            19.9|         8.72|              28.62|            1|
+--------------------+----------------+----------------+-------------+-------------------+-------------+
only showing top 5 rows


In [154]:
order_curated.count()

99441

In [156]:
duplicate_checks(order_curated, "order_id").count()

0

In [157]:
dataframe["customers"].printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [158]:
dataframe["customers"].printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [159]:
dataframe['customers'].count()

99441

In [161]:
customer_duplicate_check = duplicate_checks(dataframe["customers"],'customer_id')
customer_duplicate_check.count()

0

In [165]:
orders_without_customer = dataframe["orders"].join(dataframe["customers"],on="customer_id",how="left_anti")
orders_without_customer.count()

0

In [166]:
dataframe["customers"].select("customer_unique_id").distinct().count()

96096

In [167]:
order_curated = order_curated.join(dataframe['customers'],on="customer_id",how="left")